# SeaData End-to-End Pipeline Walkthrough

This notebook demonstrates the full SeaData workflow: data loading, training, calibrated evaluation, and visual reporting.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pathlib import Path
import pandas as pd

# Set working directory to project root
if Path.cwd().name == "notebooks":
    os.chdir("..")

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

from src.cli import build_experiment, load_raw_frame
from src.explain.report import generate_visual_report
from src.utils.io import read_json

# 1. Load Configuration
config_path = "configs/mlp_water_level.yaml"
config, experiment = build_experiment(config_path)

print(f"Experiment initialized: {experiment.name()}")
print(f"Working directory: {Path.cwd()}")

Experiment initialized: mlp_classifier
Working directory: /root/seaData/backend


In [2]:
# 2. Run Full Pipeline (Preprocess -> Train -> Evaluate -> Visualize)
frame = load_raw_frame(config)
experiment.run(frame)

print("Pipeline execution complete!")

2026-05-12 23:28:09,738 | INFO | mlp_water_level_classification | 🚀 Starting experiment: mlp_water_level_classification
2026-05-12 23:28:09,961 | INFO | mlp_water_level_classification | Generating lag features for columns: ['rainfall_mm', 'temperature_c', 'pressure_hpa']
2026-05-12 23:28:10,240 | INFO | mlp_water_level_classification | Dropped first 72 rows for lag feature warmup. Remaining rows: 43559
2026-05-12 23:28:22,233 | INFO | trainer | Training model on device: cuda
Training epochs: 100%|██████████| 150/150 [00:39<00:00,  3.80epoch/s, val_acc=0.9643, val_loss=0.0481]
2026-05-12 23:29:02,393 | INFO | calibration | Collecting logits for 2215 rows on cuda (batch_size=4096)...
Inference: 100%|██████████| 1/1 [00:00<00:00, 653.83batch/s]
2026-05-12 23:29:02,428 | INFO | mlp_water_level_classification | Applying temperature scaling and generating predictions...
2026-05-12 23:29:02,429 | INFO | calibration | Collecting logits for 17364 rows on cuda (batch_size=4096)...
Inference: 100

AssertionError: The SHAP explanations do not sum up to the model's output! This is either because of a rounding error or because an operator in your computation graph was not fully supported. If the sum difference of %f is significant compared to the scale of your model outputs, please post as a github issue, with a reproducible example so we can debug it. Used framework: pytorch - Max. diff: 0.6454850880289005 - Tolerance: 0.01

In [ ]:
# 3. Inspect Metrics
eval_path = config.paths.evaluation_json
metrics = read_json(eval_path)

print(f"Accuracy: {metrics.get('accuracy', 0):.2%}")
print(f"Event Recall: {metrics.get('event_recall', 0):.2%}")
if "mean_event_confidence" in metrics:
    print(f"Avg Event Confidence: {metrics.get('mean_event_confidence', 0):.1%}")

Accuracy: 97.61%
Event Recall: 94.78%
Avg Event Confidence: 98.2%


In [ ]:
# 4. Generate Premium Visual Report
report_dir = config.paths.evaluation_json.parent
importance_csv = report_dir / "feature_importance.csv"

if importance_csv.exists():
    importance_df = pd.read_csv(importance_csv)

    generate_visual_report(
        importance_df=importance_df,
        metrics=metrics,
        output_path=report_dir / "notebook_visual_report.pdf",
        experiment_name=experiment.name(),
        plots_dir=report_dir,
    )
    print(f"PDF Report generated: {report_dir}/notebook_visual_report.pdf")
else:
    print("Run SHAP analysis first to generate feature importance CSV.")

Run SHAP analysis first to generate feature importance CSV.
